[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/10_datenqualitaet_2.ipynb)

# Sitzung 10 — Datenqualität: Fake Reviews, Sarkasmus, Mehrsprachigkeit

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI · 09.12.2026**

Letzte Woche haben wir gesehen, dass das LLM **lügen** kann. Heute geht es um den anderen Weg, auf dem ein Dashboard falsch wird: Das Modell arbeitet ordentlich, aber die **Daten** führen es in die Irre. Gefälschte Bewertungen, Ironie, eine zweite Sprache, und am Ende die grundsätzlichste Frage: **Wer schreibt überhaupt Bewertungen?**

Das Muster, das wir heute immer wieder sehen:

```text
schlechte Eingabe  ──>  ordentliches Modell  ──>  sauberes Dashboard  ──>  falsche Entscheidung
                                                   (sieht gut aus!)
```

**Ablauf**
1. **Drei Störenfriede (Abschnitt 1):** wie viele Fakes, Sarkasmus und englische Bewertungen stecken in unseren Daten?
2. **Fakes (Abschnitt 2):** Was passiert mit den Kennzahlen, wenn der Filter aus Sitzung 4 fehlt?
3. **Sarkasmus (Abschnitt 3):** positive Wörter, negative Aussage, und warum das gerade den Trend verfälscht.
4. **Mehrsprachigkeit (Abschnitt 4):** wenn ein Werkzeug nur Deutsch kann.
5. **Eine saubere Vergleichsbasis (Abschnitt 5):** Wie gut wäre die Analyse mit perfekten Daten?
6. **Wer schreibt Bewertungen? (Abschnitt 6):** die Frage, die kein Filter beantwortet.

Heute brauchen wir den **API-Schlüssel** (Colab Secrets, wie gewohnt). Ohne Schlüssel läuft alles mit dem Mock; die Zahlen aus der Wahrheit bleiben dieselben.

### Rückblick auf Sitzung 9

```text
1. Zusammenfassung aus Rohdaten      -> klingt überzeugend, verallgemeinert Einzelfälle
2. Zusammenfassung aus Fakten        -> das LLM formuliert, rechnet nicht selbst
3. Derselbe Prompt, neuer Lauf       -> anderer Text: ein Lauf ist eine Stichprobe, kein Ergebnis
4. Jede Behauptung                   -> auf eine Zahl oder eine Bewertung zurückführen
5. Fehlklassifikation                -> Begründung lesen: Modellfehler oder unscharfe Wahrheit?
```

Sitzung 9 war die Frage: *Lügt das Modell?* Heute fragen wir: *Lügen die Daten?*

## 0. Setup und Daten

**Was passiert in der nächsten Zelle?** Wir laden zwei Fassungen der Daten: die **Rohdaten** (`export_roh`, wie in Sitzung 4 am Anfang) und die **bereinigten** Daten (`lade_sauber`, wie am Ende von Sitzung 4). Dazu hängen wir drei Marker aus der Wahrheit an: Ist eine Bewertung gefälscht, sarkastisch, englisch?

In [ ]:
!pip install anthropic --quiet

import json, re, random, urllib.request, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
pd.set_option("display.max_colwidth", 90)

URL = "https://raw.githubusercontent.com/kevisback/bda-course/main/wise2627/notebooks/radar.py"
try:
    urllib.request.urlretrieve(URL, "radar.py")
except Exception as e:
    print("Download fehlgeschlagen, nutze lokale radar.py:", e)
import radar
importlib.reload(radar)
radar.setup(backend="anthropic", model="claude-haiku-4-5")

roh, wahrheit = radar.export_roh()                 # Rohdaten, wie aus dem Export
df, _ = radar.lade_sauber()                        # bereinigt wie in Sitzung 4 (ohne Fakes, Müll, Duplikate)

df["wahr"] = wahrheit.loc[df["review_id"], "true_sentiment"].values
df["sarkasmus"] = wahrheit.loc[df["review_id"], "is_sarcastic"].values
df["englisch"] = wahrheit.loc[df["review_id"], "is_english"].values
df["sterne_f"] = df["sterne"].astype("float")      # Sterne als Kommazahl, fehlende = NaN

print(f"Rohdaten: {len(roh)} Zeilen | bereinigt: {len(df)} Bewertungen")

**Wie lese ich das?** 311 Zeilen roh, 279 nach der Bereinigung aus Sitzung 4. Die drei Marker `sarkasmus`, `englisch` und (in den Rohdaten) `is_fake` stammen aus unserem Generator. In echten Daten gibt es sie **nicht**: Dort muss man Fakes, Ironie und Sprache erst selbst erkennen. Genau deshalb sind sie heute so nützlich: Wir können messen, was sie anrichten.

## 1. Drei Störenfriede

| Störenfried | Was ist das? | Was richtet er an? |
|---|---|---|
| **Fake-Bewertung** | Werbung, Spam, gekaufte Sterne | verschiebt Kennzahlen in eine Richtung, meist nach oben |
| **Sarkasmus** | positive Wörter, negative Aussage | wird als Lob gezählt, obwohl es Kritik ist |
| **Andere Sprache** | Bewertung auf Englisch | Werkzeuge, die nur Deutsch können, sehen nichts |

**Was passiert in der nächsten Zelle?** Wir zählen, wie oft jeder Störenfried vorkommt, und zeigen je ein Beispiel.

In [ ]:
fakes_roh = roh[wahrheit.loc[roh["review_id"], "is_fake"].values]
uebersicht = pd.DataFrame([
    {"Störenfried": "Fake (in den Rohdaten)", "Anzahl": len(fakes_roh), "von": len(roh),
     "Beispiel": fakes_roh["text"].iloc[0]},
    {"Störenfried": "Sarkasmus (bereinigt)", "Anzahl": int(df["sarkasmus"].sum()), "von": len(df),
     "Beispiel": df.loc[df["sarkasmus"], "text"].iloc[0]},
    {"Störenfried": "Englisch (bereinigt)", "Anzahl": int(df["englisch"].sum()), "von": len(df),
     "Beispiel": df.loc[df["englisch"], "text"].iloc[0]},
]).set_index("Störenfried")
uebersicht["Anteil"] = (uebersicht["Anzahl"] / uebersicht["von"]).map("{:.0%}".format)
with pd.option_context("display.max_colwidth", None):
    display(uebersicht[["Anzahl", "von", "Anteil", "Beispiel"]])

**Wie lese ich das?** 12 Fakes in den Rohdaten (4 %), 25 sarkastische (9 %) und 16 englische Bewertungen (6 %) in den bereinigten Daten. Das klingt nach wenig. Aber diese Bewertungen sind nicht zufällig verteilt: Fakes sind alle positiv, Sarkasmus ist immer Kritik. Sie verschieben die Zahlen also nicht zufällig, sondern **in eine Richtung**.

**Was passiert in der nächsten Zelle?** Wie sieht das im Verhältnis zur Stimmung aus? Ein Balken pro wahrer Stimmung, darin markiert, wie viel Sarkasmus und Englisch steckt.

In [ ]:
art = np.where(df["sarkasmus"], "sarkastisch", np.where(df["englisch"], "englisch", "normal"))
zusammensetzung = pd.crosstab(df["wahr"], art, rownames=["wahre Stimmung"], colnames=["Art"]).reindex(["positive", "mixed", "neutral", "negative"])
zusammensetzung = zusammensetzung[["normal", "englisch", "sarkastisch"]]

ax = zusammensetzung.plot(kind="barh", stacked=True, color=["#d1d5db", "#2563eb", "#dc2626"], figsize=(8, 3.2), width=0.75)
ax.set_title("Bereinigte Bewertungen nach wahrer Stimmung: Wie viel ist sarkastisch oder englisch?")
ax.set_xlabel("Anzahl Bewertungen"); ax.set_ylabel(""); ax.invert_yaxis()
ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
zusammensetzung

**Wie lese ich das?** Der rote Teil sitzt komplett bei den **negativen** Bewertungen: Von 106 negativen Bewertungen sind 25 sarkastisch, fast jede vierte. Wer Sarkasmus nicht erkennt, verliert also nicht 9 % aller Bewertungen, sondern ein Viertel der Kritik. Die englischen Bewertungen (blau) verteilen sich dagegen auf positiv, gemischt und negativ.

## 2. Fakes: Was, wenn der Filter fehlt?

In Sitzung 4 habt ihr eine Regel `ist_spam` gebaut und die Fakes entfernt. Heute stellen wir die Frage andersherum: **Was hätten die 12 Fakes angerichtet**, wenn wir sie übersehen hätten?

Fakes sind Werbung: Sie geben fast immer 5 Sterne, und ein Modell, das nur nach der Stimmung fragt, liest „BESTES PRODUKT EVER!!!" als **positiv**.

**Was passiert in der nächsten Zelle?** Wir legen die 12 Fakes zu den bereinigten Daten dazu, so wie eine Pipeline ohne Spam-Filter sie zählen würde (positiv, mit ihren Sternen), und vergleichen zwei Kennzahlen.

In [ ]:
fakes = pd.DataFrame({
    "datum": pd.to_datetime(wahrheit.loc[fakes_roh["review_id"], "datum_wahr"].values),
    "sterne_f": wahrheit.loc[fakes_roh["review_id"], "sterne_wahr"].astype("float").values,
    "wahr": "positive",                                       # so würde eine Pipeline ohne Filter sie zählen
})
mit_fakes = pd.concat([df[["datum", "sterne_f", "wahr"]], fakes], ignore_index=True)

def kennzahlen(daten):
    return {"Bewertungen": len(daten), "Anteil positiv": (daten["wahr"] == "positive").mean(),
            "Ø Sterne": daten["sterne_f"].mean()}

vergleich = pd.DataFrame({"ohne Fakes (Sitzung 4)": kennzahlen(df), "mit Fakes": kennzahlen(mit_fakes)}).T
vergleich.style.format({"Anteil positiv": "{:.1%}", "Ø Sterne": "{:.2f}"})

**Wie lese ich das?** Mit den Fakes steigt der Anteil positiver Bewertungen von 38,0 % auf 40,5 %, der Sterne-Durchschnitt von 3,23 auf 3,31. Das ist nicht dramatisch, aber es sind auch nur 12 Fakes. Die wichtigere Frage: **Wie viele braucht es**, bis das Dashboard eine andere Geschichte erzählt?

**Was passiert in der nächsten Zelle?** Wir spielen das durch: 0, 10, 20, … 100 zusätzliche Fake-Bewertungen mit 5 Sternen, und zeichnen, wie sich beide Kennzahlen bewegen.

In [ ]:
zeilen = []
n_pos, n = (df["wahr"] == "positive").sum(), len(df)
sterne_summe, sterne_n = df["sterne_f"].sum(), df["sterne_f"].notna().sum()
for k in range(0, 101, 10):                                    # k Fakes mit 5 Sternen dazu
    zeilen.append({"Fakes": k, "Anteil Fakes": k / (n + k),
                   "Anteil positiv": (n_pos + k) / (n + k),
                   "Ø Sterne": (sterne_summe + 5 * k) / (sterne_n + k)})
sim = pd.DataFrame(zeilen).set_index("Fakes")

fig, (links, rechts) = plt.subplots(1, 2, figsize=(11, 3.6))
links.plot(sim.index, sim["Anteil positiv"], marker="o", color="#16a34a")
links.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); links.set_title("Anteil positiver Bewertungen")
rechts.plot(sim.index, sim["Ø Sterne"], marker="o", color="#2563eb"); rechts.set_title("Ø Sterne")
for ax in (links, rechts):
    ax.set_xlabel("zusätzliche Fake-Bewertungen"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
sim.loc[[0, 30, 60, 100]].style.format({"Anteil Fakes": "{:.0%}", "Anteil positiv": "{:.0%}", "Ø Sterne": "{:.2f}"})

**Wie lese ich das?** Mit 30 Fakes (gut 10 % der Bewertungen) steigt der Sterne-Durchschnitt von 3,23 auf 3,43, mit 100 Fakes auf 3,76. Aus „eher durchwachsen" wird „ziemlich gut". Fake-Bewertungen kosten im Netz wenige Euro pro Stück; ein Händler, der sein Produkt aufhübschen will, braucht dafür kein großes Budget.

Zwei Lehren:
- Der Spam-Filter aus Sitzung 4 ist **kein Schönheitsdetail**, sondern schützt jede Kennzahl im Dashboard.
- Eure Regel kannte genau diese zehn Fake-Texte. Echte Fälscher lernen dazu: weniger Ausrufezeichen, keine Links, glaubwürdige Sätze. Ein Filter muss **regelmäßig neu geprüft** werden, sonst schützt er gegen die Fakes vom letzten Jahr.

**Was passiert in der nächsten Zelle?** Fakes verschieben nicht nur den Durchschnitt, sondern auch den **Zeitverlauf**: Wann wurden sie geschrieben?

In [ ]:
def anteil_positiv_pro_quartal(daten):
    d = daten.dropna(subset=["datum"])
    return (d["wahr"] == "positive").groupby(d["datum"].dt.to_period("Q")).mean()

zeit = pd.DataFrame({"ohne Fakes": anteil_positiv_pro_quartal(df), "mit Fakes": anteil_positiv_pro_quartal(mit_fakes)})
zeit["Fakes im Quartal"] = fakes["datum"].dt.to_period("Q").value_counts().reindex(zeit.index, fill_value=0)

ax = zeit[["ohne Fakes", "mit Fakes"]].plot(kind="bar", color=["#9ca3af", "#16a34a"], figsize=(7, 3.4), width=0.75)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylim(0, 0.8)
ax.set_title("Anteil positiver Bewertungen pro Quartal"); ax.set_xlabel("")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
zeit.style.format({"ohne Fakes": "{:.0%}", "mit Fakes": "{:.0%}"})

**Wie lese ich das?** Die Fakes verteilen sich nicht gleichmäßig über das Jahr. In unseren Daten stammen die meisten aus der ersten Jahreshälfte, also aus der Zeit, in der es ohnehin besser lief. Mit Fakes sieht der Start noch freundlicher aus, und der Absturz danach wirkt steiler.

In echten Daten ist es oft umgekehrt: Fakes kommen in **Wellen**, etwa zum Produktstart oder nach einer schlechten Presse. Dann täuschen sie genau dort eine Erholung vor, wo man hinschaut.

> **Nicht:** „Ein paar Fakes mitteln sich schon raus."
> **Sondern:** Fakes zeigen alle in **dieselbe** Richtung und oft auf **denselben** Zeitraum. Sie mitteln sich nicht raus, sie addieren sich.

## 3. Sarkasmus

> „Super, schon nach drei Tagen kaputt. Echt klasse Qualität."

Ein Mensch erkennt sofort: Das ist Kritik. Ein Werkzeug, das Wörter zählt, sieht „super" und „klasse", zwei positive Wörter gegen ein negatives.

**Was passiert in der nächsten Zelle?** Wir bauen so ein Werkzeug, eine einfache **Wortregel** wie in Sitzung 2: positive und negative Wörter zählen, die Mehrheit entscheidet. Dann messen wir, wie oft sie richtig liegt, getrennt für normale, sarkastische und englische Bewertungen.

In [ ]:
POS_WORTE = ["super", "toll", "klasse", "perfekt", "wunderbar", "herrlich", "fantastisch", "sensationell",
             "bravo", "hervorragend", "top", "bequem", "begeistert", "gelungen", "empfehlen", "gut",
             "zufrieden", "liebe", "satte", "sofort", "ganzen tag", "kino", "traum"]
NEG_WORTE = ["kaputt", "stürzt", "rauscht", "teuer", "billig", "leer", "bricht", "matschig", "enttäusch",
             "schlecht", "finger weg", "frustriert", "ärgerlich", "reklamiert", "nie wieder", "lädt nicht",
             "reagiert kaum", "fallen", "fällt", "tot", "nichts", "schade", "vorsicht", "zurückgeschickt",
             "raus", "rauschen", "weg"]

def wortregel(text):
    t = text.lower()
    pos = sum(w in t for w in POS_WORTE)
    neg = sum(w in t for w in NEG_WORTE)
    if pos == 0 and neg == 0:
        return "neutral"                                       # kein bekanntes Wort gefunden
    if pos > neg:
        return "positive"
    if neg > pos:
        return "negative"
    return "mixed"

df["regel"] = df["text"].map(wortregel)
df["art"] = art
treffer_regel = (df["regel"] == df["wahr"]).groupby(df["art"]).agg(["mean", "count"])
treffer_regel.columns = ["Wortregel richtig", "Bewertungen"]
treffer_regel.style.format({"Wortregel richtig": "{:.0%}"})

**Wie lese ich das?** Bei normalen Bewertungen liegt die Wortregel in 79 % der Fälle richtig. Das ist für so ein einfaches Werkzeug ordentlich. Bei sarkastischen Bewertungen sind es nur **24 %**, bei englischen **0 %**. Das Werkzeug ist nicht schlechter geworden, die **Daten** sind andere.

**Was passiert in der nächsten Zelle?** Was genau sagt die Wortregel zu den sarkastischen Bewertungen? Jeder Text einmal, mit Sternen.

In [ ]:
sarkastisch = df[df["sarkasmus"]].drop_duplicates("text")[["text", "regel", "sterne"]]
with pd.option_context("display.max_colwidth", None):
    display(sarkastisch)
print("Wortregel bei allen 25 sarkastischen Bewertungen:", df.loc[df["sarkasmus"], "regel"].value_counts().to_dict())

**Wie lese ich das?** Die Wortregel nennt die meisten sarkastischen Bewertungen *gemischt* oder *positiv*, nur 6 von 25 *negativ*. Schaut auf die Sterne: 1 oder 2, wo sie angegeben sind. Die Kunden selbst haben eindeutig bewertet; nur ihre Wörter sagen das Gegenteil.

Aber: Bei 10 der 25 fehlen die Sterne. Das kennt ihr aus Sitzung 4: Negative Bewertungen haben besonders oft keine Sterne. Die Störenfriede treffen sich.

**Was passiert in der nächsten Zelle?** Was macht das mit der Kennzahl, die Lena in Sitzung 7 am meisten interessiert hat, dem Anteil negativer Bewertungen pro Quartal? Wir vergleichen die Wahrheit mit einer Pipeline, die **nur** beim Sarkasmus Fehler macht (alles andere nehmen wir aus der Wahrheit).

In [ ]:
mit_datum = df.dropna(subset=["datum"]).copy()
mit_datum["quartal"] = mit_datum["datum"].dt.to_period("Q")
nur_sarkasmus_falsch = mit_datum["wahr"].where(~mit_datum["sarkasmus"], mit_datum["regel"])

quartal = pd.DataFrame({
    "wahr": (mit_datum["wahr"] == "negative").groupby(mit_datum["quartal"]).mean(),
    "Sarkasmus nicht erkannt": (nur_sarkasmus_falsch == "negative").groupby(mit_datum["quartal"]).mean(),
    "sarkastische Bewertungen": mit_datum["sarkasmus"].groupby(mit_datum["quartal"]).sum(),
})
ax = quartal[["wahr", "Sarkasmus nicht erkannt"]].plot(kind="bar", color=["#dc2626", "#fca5a5"], figsize=(7, 3.4), width=0.75)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylim(0, 0.8)
ax.set_title("Anteil negativer Bewertungen pro Quartal"); ax.set_xlabel("")
ax.legend(frameon=False, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
quartal.style.format({"wahr": "{:.0%}", "Sarkasmus nicht erkannt": "{:.0%}"})

**Wie lese ich das?** Im ersten Quartal macht der Sarkasmus 6 Prozentpunkte aus (20 % statt 14 %), im dritten Quartal **14 Prozentpunkte** (68 % statt 55 %). Warum dort mehr? Sarkasmus ist Kritik, und Kritik häuft sich in unseren Daten im Sommer: 13 der 24 sarkastischen Bewertungen mit Datum stammen aus dem dritten Quartal.

Das heißt: Ein Werkzeug, das Sarkasmus übersieht, macht nicht überall denselben kleinen Fehler. Es macht den **größten Fehler genau dort, wo es schlechter wird**, und lässt den Absturz harmloser aussehen, als er ist.

> **Nicht:** „9 % der Bewertungen sind sarkastisch, das ist ein kleiner Fehler."
> **Sondern:** Sarkasmus ist immer Kritik. Er verschiebt die Kennzahl in eine Richtung, und am stärksten dort, wo es am meisten Kritik gibt.

**Was passiert in der nächsten Zelle?** Kann das LLM, was die Wortregel nicht kann? Wir lassen es alle 279 Bewertungen einordnen, mit der Batch-Funktion aus Sitzung 3, und vergleichen wieder getrennt nach normal, sarkastisch, englisch.

In [ ]:
df["llm"] = radar.classify_many(df["review_id"], df["text"])           # Batches wie in Sitzung 3

treffer = pd.DataFrame({
    "Wortregel": (df["regel"] == df["wahr"]).groupby(df["art"]).mean(),
    "LLM": (df["llm"] == df["wahr"]).groupby(df["art"]).mean(),
    "Bewertungen": df["art"].value_counts(),
}).reindex(["normal", "sarkastisch", "englisch"])
print("Backend:", radar.BACKEND, "| Verbrauch:", radar.verbrauch())
treffer.style.format({"Wortregel": "{:.0%}", "LLM": "{:.0%}"})

**Wie lese ich das?** Mit dem echten Modell liegt das LLM bei sarkastischen und englischen Bewertungen typischerweise **weit** über der Wortregel: Es versteht, dass „Ganz großes Kino, der Akku hält sagenhafte 40 Minuten" Kritik ist, und es liest Englisch. (Im Mock sieht das anders aus, der Mock zählt nur Wörter.)

Hier macht das Modell also wirklich einen Unterschied. Aber schaut, **wie** wir das herausgefunden haben: Insgesamt liegen beide Werkzeuge gar nicht so weit auseinander, weil 238 der 279 Bewertungen normal sind. Erst die **Aufteilung nach Datenart** zeigt, wo der Unterschied steckt.

**Was passiert in der nächsten Zelle?** Wo liegt das LLM trotzdem daneben? Die sarkastischen Bewertungen, die es nicht als negativ erkannt hat.

In [ ]:
verpasst = df[df["sarkasmus"] & (df["llm"] != "negative")].drop_duplicates("text")
print(f"{(df['sarkasmus'] & (df['llm'] != 'negative')).sum()} von {df['sarkasmus'].sum()} sarkastischen Bewertungen nicht als negativ erkannt")
with pd.option_context("display.max_colwidth", None):
    display(verpasst[["text", "llm", "sterne"]].head(6))

**Wie lese ich das?** Wenn hier Bewertungen stehen, lohnt sich ein genauer Blick: Ist der Sarkasmus wirklich eindeutig, oder könnte man „Fantastisch leise – weil nach zwei Tagen einfach tot" auch als gemischt lesen? Manche Fehler sind Modellfehler, manche zeigen, dass auch unsere Wahrheit eine Entscheidung ist (Sitzung 1). Steht hier nichts, hat das Modell alle erkannt, bei diesen 25 Bewertungen.

## 4. Mehrsprachigkeit

16 unserer Bewertungen sind auf Englisch. Für das LLM ist das kein Problem. Für jedes Werkzeug, das mit **deutschen Wortlisten** arbeitet, schon: Die Wortregel oben, die Stichwort-Themen aus Sitzung 5, ein deutscher Spam-Filter.

**Was passiert in der nächsten Zelle?** Was sagen Wortregel und LLM zu den englischen Bewertungen? Jeder Text einmal.

In [ ]:
englisch = df[df["englisch"]].drop_duplicates("text")[["text", "wahr", "regel", "llm"]]
with pd.option_context("display.max_colwidth", None):
    display(englisch)
print("Wortregel bei allen 16 englischen Bewertungen:", df.loc[df["englisch"], "regel"].value_counts().to_dict())

**Wie lese ich das?** Die Wortregel nennt 12 der 16 englischen Bewertungen *neutral*, weil sie kein einziges deutsches Wort findet. Die übrigen 4 nennt sie *positiv*, und zwar ausgerechnet „Stopped working after a week, very disappointed.": In „S**top**ped" steckt das Wort „top". **Keine** einzige englische Bewertung wird als negativ erkannt, obwohl 7 negativ sind.

Das Muster ist dasselbe wie beim Sarkasmus: Der Fehler ist nicht zufällig, er zeigt **in eine Richtung**. Englische Kritik verschwindet, und der Anteil negativer Bewertungen sinkt.

**Was passiert in der nächsten Zelle?** Bevor man ein Werkzeug wählt, sollte man wissen, welche Sprachen in den Daten stecken. Eine einfache Prüfung: Wie viele typisch englische Wörter enthält ein Text?

In [ ]:
ENGLISCHE_WOERTER = {"the", "and", "is", "it", "after", "very", "with", "best", "great", "but", "i", "have"}

def wirkt_englisch(text):
    woerter = re.findall(r"[a-zäöüß]+", text.lower())
    return sum(w in ENGLISCHE_WOERTER for w in woerter) >= 2      # mindestens zwei englische Wörter

df["sprache_erkannt"] = df["text"].map(lambda t: "englisch" if wirkt_englisch(t) else "deutsch")
print(pd.crosstab(df["englisch"].map({True: "wirklich englisch", False: "wirklich deutsch"}), df["sprache_erkannt"]), "\n")
pd.crosstab(df["quelle"], df["sprache_erkannt"])

**Wie lese ich das?** Oben: Die einfache Prüfung erkennt die englischen Bewertungen zuverlässig, ohne Fehlalarm bei deutschen. Für die Frage *„Wie viel Englisch steckt in meinen Daten?"* reicht so eine grobe Zählung völlig.

Unten: Die englischen Bewertungen verteilen sich auf **alle** Quellen, auch auf die deutsche App und den Webshop. Wer annimmt „Englisch kommt nur aus der US-App", filtert falsch. Ein Datensatz verrät seine Sprachen erst, wenn man nachzählt.

> **Nicht:** „Unsere Kunden schreiben Deutsch."
> **Sondern:** Erst nachzählen, welche Sprachen vorkommen, dann das Werkzeug wählen. Und für jede Sprache einzeln prüfen, wie gut es funktioniert.

## 5. Eine saubere Vergleichsbasis

Bisher haben wir gezeigt, wo Werkzeuge an schwierigen Daten scheitern. Zur Gegenprobe brauchen wir das Gegenteil: **perfekte Daten.** Kurze, eindeutige, deutsche Bewertungen ohne Ironie. Wie gut wären Wortregel und LLM dort?

Solche Daten lassen wir uns vom LLM schreiben. Das ist ein übliches Vorgehen, um ein Werkzeug zu testen, und gleichzeitig eine Falle, wie wir gleich sehen.

**Was passiert in der nächsten Zelle?** Das LLM schreibt 20 eindeutige Bewertungen, je 10 positive und negative, mit Label. Ohne Schlüssel bauen wir sie aus Satzbausteinen zusammen.

In [ ]:
BASIS_PROMPT = """Schreibe 20 kurze, eindeutige Produktbewertungen zu kabellosen Ohrhörern auf Deutsch:
10 klar positive und 10 klar negative. Keine Ironie, keine gemischten Bewertungen, je ein Satz.
Antworte nur mit JSON: [{"text": "...", "label": "positive" oder "negative"}, ...]"""

def baue_basis():
    if radar.BACKEND != "mock":
        antwort = radar.mit_retry(radar.call_llm, BASIS_PROMPT, temperature=0, max_tokens=2000)["text"]
        try:
            liste = json.loads(re.search(r"\[.*\]", antwort, re.DOTALL).group(0))
            liste = [x for x in liste if x.get("label") in ("positive", "negative") and x.get("text")]
            if len(liste) >= 10:
                return pd.DataFrame(liste)
        except (AttributeError, json.JSONDecodeError):
            pass
        print("⚠️ Antwort nicht lesbar, nutze Satzbausteine.")
    rng = random.Random(1)                                     # Ersatz: aus Satzbausteinen
    pos = [{"text": f"{rng.choice(radar.POS_OPENERS)} {p}.", "label": "positive"} for p in rng.sample(radar.POS, 10)]
    neg = [{"text": f"{rng.choice(radar.NEG_OPENERS)} {n}.", "label": "negative"} for n in rng.sample(radar.NEG, 10)]
    return pd.DataFrame(pos + neg)

basis = baue_basis()
with pd.option_context("display.max_colwidth", None):
    display(basis.sample(6, random_state=0))

**Wie lese ich das?** Saubere, eindeutige Sätze. Genau so stellt man sich Bewertungen vor, wenn man ein Werkzeug baut. Echte Bewertungen sehen anders aus, wie die letzten drei Abschnitte gezeigt haben.

**Was passiert in der nächsten Zelle?** Wortregel und LLM auf der sauberen Basis, im Vergleich zu den drei Arten echter Bewertungen.

In [ ]:
basis["regel"] = basis["text"].map(wortregel)
basis["llm"] = radar.classify_many([f"B{i:03d}" for i in range(len(basis))], basis["text"])

vergleich_basis = pd.concat([
    pd.DataFrame({"Wortregel": [(basis["regel"] == basis["label"]).mean()],
                  "LLM": [(basis["llm"] == basis["label"]).mean()],
                  "Bewertungen": [len(basis)]}, index=["saubere Basis"]),
    treffer,
])
ax = vergleich_basis[["Wortregel", "LLM"]].plot(kind="bar", color=["#9ca3af", "#2563eb"], figsize=(8, 3.6), width=0.75)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylim(0, 1.05)
ax.set_title("Wie oft liegt das Werkzeug richtig? Saubere Basis gegen echte Daten"); ax.set_xlabel("")
ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
vergleich_basis.style.format({"Wortregel": "{:.0%}", "LLM": "{:.0%}"})

**Wie lese ich das?** Auf der sauberen Basis sehen **beide** Werkzeuge hervorragend aus, auch die simple Wortregel. Hätten wir nur dort getestet, wäre unser Urteil gewesen: „Die Wortregel reicht, das LLM brauchen wir nicht." Erst die echten Daten zeigen, wo die Wortregel zusammenbricht.

Das ist eine der häufigsten Fallen in der Praxis: Ein Werkzeug wird an Beispielen getestet, die jemand **sich ausgedacht** hat, ob ein Mensch oder ein LLM. Solche Beispiele sind sauber, eindeutig und ohne die Eigenheiten echter Kunden. Die Testergebnisse sind dann zu gut.

> **Nicht:** „Unser Werkzeug hat 95 % Trefferquote." (gemessen an erfundenen Beispielen)
> **Sondern:** Testen an einer **Stichprobe echter Daten**, und das Ergebnis getrennt nach den schwierigen Arten angeben.

Und das ist die eigentliche Lehre der ersten Hälfte heute: Ob die Zahlen stimmen, entscheidet oft nicht die Klugheit des Modells, sondern die **Qualität und Zusammensetzung der Daten**.

✏️ **Eure Aufgabe 1: Sarkasmus finden, ohne die Wahrheit**

In echten Daten gibt es keine Spalte `sarkasmus`. Aber es gibt einen Hinweis: Bei Sarkasmus **widersprechen sich Text und Sterne**. Die Wörter klingen nicht negativ, aber der Kunde gibt nur einen Stern.

1. Füllt die Lücken `...`: Sucht alle Bewertungen, bei denen die Wortregel **nicht** „negative" sagt und die Sterne genau **1** sind.
2. Wie viele der Kandidaten sind wirklich sarkastisch? Wie viele sarkastische Bewertungen findet die Regel **nicht**, und warum?

Die Lösung steht in der Zelle danach. Erst selbst versuchen.

In [ ]:
# Eure Lösung: Lücken ... füllen
# (try/except nur, damit „Alle ausführen" hier nicht stoppt. Die Lösungszelle darunter braucht es nicht.)
try:
    LABEL = ...                                          # welches Label sagt die Wortregel NICHT?
    STERNE = ...                                         # wie viele Sterne?
    if LABEL is ... or STERNE is ...:
        raise ValueError("Lücken noch nicht gefüllt")
    kandidaten = df[(df["regel"] != LABEL) & (df["sterne_f"] == STERNE)]
    print(f"{len(kandidaten)} Kandidaten, davon wirklich sarkastisch: {kandidaten['sarkasmus'].sum()}")
    display(kandidaten[["text", "regel", "sterne", "sarkasmus"]])
except Exception as e:
    print("Noch nicht fertig oder ein Fehler:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
kandidaten = df[(df["regel"] != "negative") & (df["sterne_f"] == 1)]
gefunden = kandidaten["sarkasmus"].sum()
print(f"{len(kandidaten)} Kandidaten, davon wirklich sarkastisch: {gefunden}")
print(f"Gefunden: {gefunden} von {df['sarkasmus'].sum()} sarkastischen Bewertungen | "
      f"davon ohne Sterne (unauffindbar): {df.loc[df['sarkasmus'], 'sterne'].isna().sum()}")
with pd.option_context("display.max_colwidth", None):
    display(kandidaten[["text", "regel", "sterne", "sarkasmus"]].drop_duplicates("text"))

**Wie lese ich das?** 13 Kandidaten, davon 10 wirklich sarkastisch: Der Widerspruch zwischen Text und Sternen ist ein **sehr treffsicheres** Signal. Und die drei anderen sind auch nicht falsch gefunden: zweimal eine englische Beschwerde, die die Wortregel nicht versteht, und einmal „Nicht zu empfehlen", das sie wegen „empfehlen" nicht als negativ zählt. Das Signal findet also allgemein Bewertungen, bei denen **das Werkzeug** danebenliegt.

Aber es findet nur 10 von 25 sarkastischen. Die anderen haben entweder 2 Sterne (dann wird es mit gemischten Bewertungen verwechselt) oder **gar keine Sterne**, und das betrifft 10 der 25. Wieder treffen sich zwei Störenfriede: Wo die Kritik am deutlichsten ist, fehlen die Sterne am häufigsten.

In der Praxis kombiniert man deshalb mehrere Signale: Widerspruch Text gegen Sterne, das LLM, und eine Stichprobe, die ein Mensch liest.

## 6. Wer schreibt überhaupt Bewertungen?

Bisher ging es um **falsche** Bewertungen: gefälscht, ironisch, unverstanden. Jetzt die unbequemere Frage: Was, wenn jede einzelne Bewertung korrekt ist, aber die **falschen Leute** schreiben?

Denkt an euch selbst: Wann habt ihr zuletzt eine Bewertung geschrieben? Meistens, wenn etwas **richtig schiefging**, oder wenn es außergewöhnlich gut war. Wer einfach zufrieden ist, schreibt selten etwas.

```text
alle Kunden            ──>  wer schreibt eine Bewertung?  ──>  was wir sehen
(die eigentliche Frage)     (hängt von der Zufriedenheit ab)   (die Bewertungen)
```

**Was passiert in der nächsten Zelle?** Wir simulieren 10.000 Kunden: 70 % zufrieden, 20 % neutral, 10 % unzufrieden. Jeder Unzufriedene schreibt mit 30 % Wahrscheinlichkeit eine Bewertung, jeder Zufriedene nur mit 6 %, jeder Neutrale mit 3 %.

In [ ]:
rng = np.random.default_rng(0)
N = 10_000
kunden = rng.choice(["zufrieden", "neutral", "unzufrieden"], size=N, p=[0.70, 0.20, 0.10])
P_BEWERTUNG = {"zufrieden": 0.06, "neutral": 0.03, "unzufrieden": 0.30}     # wer schreibt wie oft?
schreibt = rng.random(N) < np.vectorize(P_BEWERTUNG.get)(kunden)

stufen = ["zufrieden", "neutral", "unzufrieden"]
farben = ["#16a34a", "#9ca3af", "#dc2626"]
anteile = pd.DataFrame({
    f"alle Kunden (n = {N})": pd.Series(kunden).value_counts(normalize=True),
    f"die eine Bewertung schreiben (n = {schreibt.sum()})": pd.Series(kunden[schreibt]).value_counts(normalize=True),
}).reindex(stufen).T

ax = anteile.plot(kind="barh", stacked=True, color=farben, figsize=(9, 2.8), width=0.7)
for i, zeile in enumerate(anteile.values):
    links = 0
    for wert in zeile:
        if wert > 0.05:
            ax.annotate(f"{wert:.0%}", (links + wert / 2, i), ha="center", va="center", color="white", fontsize=10)
        links += wert
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_xlim(0, 1)
ax.set_title("Dieselben Kunden, zwei Blickwinkel"); ax.invert_yaxis()
ax.legend(frameon=False, bbox_to_anchor=(1, 1), loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Oben die Wahrheit: 10 % der Kunden sind unzufrieden. Unten, was in den Bewertungen ankommt: rund **36 %** unzufrieden. Mehr als dreimal so viel, ohne eine einzige gefälschte oder falsch verstandene Bewertung. Jede einzelne Bewertung ist ehrlich, nur die **Auswahl** ist verzerrt.

Kommt euch die Größenordnung bekannt vor? In unseren Daten sind 38 % der Bewertungen negativ. Was das über die Zufriedenheit **aller** Kunden der Nimbus Q2 sagt, wissen wir nicht. Das ist **Selektionsverzerrung**: Die Daten beantworten eine andere Frage als die, die wir stellen.

**Was passiert in der nächsten Zelle?** Wie stark hängt das Ergebnis davon ab, wie viel lieber Unzufriedene schreiben? Wir lassen den Anteil echter Unzufriedener bei 10 % und ändern nur ihre Schreib-Wahrscheinlichkeit.

In [ ]:
zeilen = []
for p_unzufrieden in [0.03, 0.06, 0.10, 0.20, 0.30, 0.50]:
    p = {"zufrieden": 0.06, "neutral": 0.03, "unzufrieden": p_unzufrieden}
    erwartet = {k: anteil * p[k] for k, anteil in [("zufrieden", 0.7), ("neutral", 0.2), ("unzufrieden", 0.1)]}
    zeilen.append({"Unzufriedene schreiben mit": p_unzufrieden,
                   "so oft wie Zufriedene": p_unzufrieden / 0.06,
                   "Anteil unzufrieden in den Bewertungen": erwartet["unzufrieden"] / sum(erwartet.values())})
kurve = pd.DataFrame(zeilen)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(kurve["so oft wie Zufriedene"], kurve["Anteil unzufrieden in den Bewertungen"], marker="o", color="#dc2626",
        label="Anteil unzufrieden in den Bewertungen")
ax.axhline(0.10, color="#111827", ls="--", label="wahrer Anteil unzufriedener Kunden (10 %)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylim(0, 0.6)
ax.set_xlabel("Unzufriedene schreiben x-mal so oft wie Zufriedene")
ax.set_title("Gleiche Kunden, andere Schreiblust"); ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()
kurve.style.format({"Unzufriedene schreiben mit": "{:.0%}", "so oft wie Zufriedene": "{:.1f}",
                    "Anteil unzufrieden in den Bewertungen": "{:.0%}"})

**Wie lese ich das?** Die gestrichelte Linie bleibt immer bei 10 %: Die Kunden sind in allen Fällen dieselben. Nur wenn Unzufriedene **genauso oft** schreiben wie Zufriedene (x = 1), stimmen Bewertungen und Wirklichkeit ungefähr überein. Schreiben sie fünfmal so oft, sind schon 38 % der Bewertungen unzufrieden.

Wie oft Unzufriedene tatsächlich häufiger schreiben, steht in keinem Datensatz. Man kann es nur von außen schätzen, etwa mit einer Umfrage unter **zufällig ausgewählten** Käufern.

> **Nicht:** „38 % unserer Kunden sind unzufrieden."
> **Sondern:** „38 % der **Bewertungen** sind negativ." Wer eine Aussage über alle Kunden will, braucht eine zweite Quelle: Retouren, Support-Anfragen, eine Zufallsumfrage.

**Was passiert in der nächsten Zelle?** Noch heimtückischer wird es, wenn sich die Auswahl **mit der Zeit ändert**. Angenommen, der Shop beginnt am 1. Juli, jeden Käufer per E-Mail um eine Bewertung zu bitten. Danach schreiben Zufriedene viel häufiger. Die Kunden selbst bleiben genau gleich zufrieden.

In [ ]:
monate = pd.period_range("2026-01", "2026-09", freq="M")
zeilen = []
for m in monate:
    p_zufrieden = 0.06 if m < pd.Period("2026-07", "M") else 0.25        # ab Juli: E-Mail mit Bitte um Bewertung
    p = {"zufrieden": p_zufrieden, "neutral": 0.03, "unzufrieden": 0.30}
    erwartet = {k: a * p[k] for k, a in [("zufrieden", 0.7), ("neutral", 0.2), ("unzufrieden", 0.1)]}
    zeilen.append({"Monat": str(m), "Anteil unzufrieden in den Bewertungen": erwartet["unzufrieden"] / sum(erwartet.values())})
verlauf = pd.DataFrame(zeilen).set_index("Monat")

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(verlauf.index, verlauf.iloc[:, 0], marker="o", color="#dc2626", label="Anteil unzufrieden in den Bewertungen")
ax.axhline(0.10, color="#111827", ls="--", label="wahrer Anteil (immer 10 %)")
ax.axvline(5.5, color="#6b7280", ls=":"); ax.annotate("ab hier: E-Mail\nmit Bitte um Bewertung", (6.1, 0.27), fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylim(0, 0.6)
ax.set_title('Eine „Verbesserung“, die keine ist'); ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Ab Juli fällt der Anteil unzufriedener Bewertungen von 38 % auf 14 %. Im Dashboard sähe das aus wie ein großer Erfolg. Tatsächlich ist **kein einziger Kunde** zufriedener geworden; es schreiben nur andere.

Denkt an Sitzung 7: Dort haben wir gefragt, ob das App-Update den Absturz erklärt. Mit dieser Simulation im Kopf gibt es eine weitere Frage, die man stellen muss, bevor man einem Trend glaubt: **Hat sich geändert, wer schreibt?** Neue Bewertungs-Mails, ein neues Shop-Design, eine Rabattaktion für Bewertungen: All das verschiebt die Kennzahl, ohne dass sich am Produkt etwas ändert.

> 💡 **Kernpunkt:** Kein Filter und kein Modell korrigiert eine verzerrte Auswahl. Man muss sie **kennen**, und die Aussage auf das beschränken, was die Daten tragen.

✏️ **Eure Aufgabe 2: Wie viel verschiebt ein Filter?**

Angenommen, unser Team hat nur die Wortregel und entscheidet: *„Englische Bewertungen versteht unser Werkzeug nicht, die werfen wir raus."* Ist das besser oder schlechter, als sie mitzuzählen?

1. Füllt die Lücken `...`: welches Label gezählt wird, und eine Maske, die nur die nicht-englischen Bewertungen auswählt (Tipp: `~df["englisch"]`).
2. Vergleicht drei Fälle mit der Wahrheit (38 %): Wortregel auf **allen** Bewertungen, Wortregel **ohne Englisch**, und Wortregel ohne Englisch **und** mit korrigiertem Sarkasmus.

Die Lösung steht in der Zelle danach.

In [ ]:
# Eure Lösung: Lücken ... füllen
try:
    ZIEL_LABEL = ...                                     # welches Label zählen wir?
    OHNE_ENGLISCH = ...                                  # Maske: True für alle nicht-englischen Bewertungen
    if ZIEL_LABEL is ... or OHNE_ENGLISCH is ...:
        raise ValueError("Lücken noch nicht gefüllt")

    def anteil_negativ(labels, maske):
        return (labels[maske] == ZIEL_LABEL).mean()

    alle = np.ones(len(df), dtype=bool)
    print("Wahrheit:                ", f"{anteil_negativ(df['wahr'], alle):.1%}")
    print("Wortregel, alle:         ", f"{anteil_negativ(df['regel'], alle):.1%}")
    print("Wortregel, ohne Englisch:", f"{anteil_negativ(df['regel'], OHNE_ENGLISCH):.1%}")
except Exception as e:
    print("Noch nicht fertig oder ein Fehler:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
def anteil_negativ(labels, maske):
    return (labels[maske] == "negative").mean()

alle = np.ones(len(df), dtype=bool)
ohne_englisch = ~df["englisch"].values
regel_sarkasmus_korrigiert = df["regel"].where(~df["sarkasmus"], "negative")   # als hätten wir Sarkasmus erkannt

faelle = pd.DataFrame({
    "Anteil negativ": [anteil_negativ(df["wahr"], alle),
                       anteil_negativ(df["regel"], alle),
                       anteil_negativ(df["regel"], ohne_englisch),
                       anteil_negativ(regel_sarkasmus_korrigiert, ohne_englisch)],
    "Bewertungen": [len(df), len(df), ohne_englisch.sum(), ohne_englisch.sum()],
}, index=["Wahrheit", "Wortregel, alle", "Wortregel, ohne Englisch", "Wortregel, ohne Englisch, Sarkasmus erkannt"])
faelle["Abweichung"] = faelle["Anteil negativ"] - faelle.loc["Wahrheit", "Anteil negativ"]
faelle.style.format({"Anteil negativ": "{:.1%}", "Abweichung": "{:+.1%}"})

**Wie lese ich das?** Die Wortregel auf allen Bewertungen liegt mit 35,8 % gut 2 Punkte unter der Wahrheit. Ohne Englisch landet sie bei 38,0 %, **genau** auf der Wahrheit. Ein perfektes Werkzeug? Nein, reines Glück: Ihre Fehler gleichen sich aus. Sarkasmus wird zu selten als negativ gezählt, manche gemischte Bewertung zu oft. Sobald wir einen Fehler beheben und den Sarkasmus erkennen, springt der Wert auf 45,2 %, **7 Punkte über** die Wahrheit, weil jetzt nur noch die anderen Fehler übrig sind.

Die Lehre: Eine Gesamtzahl, die zufällig nah an der Wahrheit liegt, beweist nicht, dass das Werkzeug funktioniert. Fehler, die sich ausgleichen, gleichen sich bei einem anderen Ausschnitt (einem Quartal, einer Quelle) **nicht** mehr aus, wie die Quartale in Abschnitt 3 gezeigt haben.

## Zusammenfassung

```text
1. Fake-Bewertungen                    -> zeigen alle in dieselbe Richtung; mitteln sich nicht raus.
                                          Der Filter aus Sitzung 4 schützt jede Kennzahl, muss aber gepflegt werden
2. Sarkasmus                           -> immer Kritik; ein Werkzeug, das ihn übersieht, verharmlost
                                          gerade dort, wo es schlechter wird
3. Andere Sprache                      -> erst nachzählen, dann Werkzeug wählen; je Sprache prüfen
4. Test an sauberen Beispielen         -> zu optimistisch; an echten Daten testen, getrennt nach Datenart
5. Widerspruch Text gegen Sterne       -> treffsicheres Signal, aber fehlende Sterne begrenzen es
6. Wer schreibt Bewertungen?           -> Selektionsverzerrung: „38 % der Bewertungen", nicht „der Kunden";
                                          ändert sich, wer schreibt, ändert sich der Trend
```

Und der Satz, der über allem steht: **Ob die Zahlen stimmen, entscheidet oft die Datenqualität, nicht die Klugheit des Modells.**

## Zum Weiterprobieren (freiwillig)

1. **Bessere Wortregel:** Ergänzt `NEG_WORTE` um „nicht zu empfehlen" (statt dass „empfehlen" positiv zählt). Wie ändert sich die Trefferquote bei normalen Bewertungen?
2. **Englische Wortliste:** Gebt der Wortregel englische Wörter („great", „disappointed", „cheap" …). Wie weit kommt ihr bei den 16 englischen Bewertungen?
3. **Schwierige Basis:** Bittet das LLM um 20 **sarkastische** Bewertungen mit Label und testet Wortregel und LLM darauf. Sind erfundene schwierige Beispiele so schwierig wie echte?
4. **Ihr seid der Shop:** Ändert in der letzten Simulation die Wahrscheinlichkeit für Unzufriedene ab Juli statt für Zufriedene (z. B. ein neuer, umständlicher Rückgabeprozess). Was zeigt das Dashboard dann?

## Ausblick

**Nächste Woche (16.12):** Kosten, Latenz, Recht und Datenschutz. Was kostet die Pipeline für 100.000 Bewertungen, wie lange dauert sie, und was müsst ihr klären, bevor ihr echte Kundendaten an ein LLM schickt? Zwei Fragen, die in jedem echten Projekt kommen: *Dürfen wir diese Daten überhaupt verwenden?* und *Was sagt die DSGVO dazu?*